# Phase 6: Advanced Analysis

This notebook creates the sales forecast, market-basket results, and product profitability groups used in Power BI.

## 1. Import libraries and load the cleaned dataset

In [ ]:
from collections import Counter
from itertools import combinations
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression

project_folder = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
processed_folder = project_folder / 'data' / 'processed'
sales = pd.read_csv(processed_folder / 'cleaned_sales.csv', parse_dates=['Order Date'])

print('Rows loaded:', len(sales))

## 2. Prepare monthly sales

In [ ]:
monthly_sales = (
    sales.set_index('Order Date')['Sales USD']
    .resample('MS')
    .sum()
    .reset_index()
    .rename(columns={'Order Date': 'Month', 'Sales USD': 'Actual Sales'})
)

monthly_sales['Month Number'] = range(len(monthly_sales))
display(monthly_sales.tail())

## 3. Forecast the next six months with a three-month moving average

In [ ]:
moving_average_values = monthly_sales['Actual Sales'].tolist()
moving_average_forecast = []

for _ in range(6):
    prediction = sum(moving_average_values[-3:]) / 3
    moving_average_forecast.append(prediction)
    moving_average_values.append(prediction)

## 4. Forecast the next six months with linear regression

In [ ]:
model = LinearRegression()
model.fit(monthly_sales[['Month Number']], monthly_sales['Actual Sales'])

future_month_numbers = pd.DataFrame({
    'Month Number': range(len(monthly_sales), len(monthly_sales) + 6)
})
linear_regression_forecast = model.predict(future_month_numbers)

## 5. Create and save the forecast dataset

In [ ]:
future_months = pd.date_range(
    start=monthly_sales['Month'].max() + pd.offsets.MonthBegin(1),
    periods=6,
    freq='MS',
)

sales_forecast = pd.DataFrame({
    'Month': future_months,
    'Moving Average Forecast': moving_average_forecast,
    'Linear Regression Forecast': linear_regression_forecast,
    'Horizon Month': range(1, 7),
    'Included in 3-Month Forecast': ['Yes'] * 3 + ['No'] * 3,
})

sales_forecast[['Moving Average Forecast', 'Linear Regression Forecast']] = (
    sales_forecast[['Moving Average Forecast', 'Linear Regression Forecast']].round(2)
)

sales_forecast.to_csv(processed_folder / 'sales_forecast.csv', index=False, date_format='%Y-%m-%d')
display(sales_forecast)

## 6. Find products purchased together

In [ ]:
products_by_order = (
    sales.groupby('Order Number')['Product Name']
    .apply(lambda products: sorted(set(products)))
)

product_order_counts = Counter()
pair_order_counts = Counter()

for products in products_by_order:
    product_order_counts.update(products)
    pair_order_counts.update(combinations(products, 2))

total_orders = len(products_by_order)
basket_rows = []

for product_pair, pair_count in pair_order_counts.items():
    product_a, product_b = product_pair
    support = pair_count / total_orders
    confidence = pair_count / product_order_counts[product_a]
    product_b_support = product_order_counts[product_b] / total_orders
    lift = confidence / product_b_support

    basket_rows.append({
        'Product A': product_a,
        'Product B': product_b,
        'Orders Together': pair_count,
        'Support %': round(support * 100, 4),
        'Confidence %': round(confidence * 100, 2),
        'Lift': round(lift, 2),
    })

market_basket = (
    pd.DataFrame(basket_rows)
    .sort_values(['Orders Together', 'Lift'], ascending=False)
    .head(50)
)

market_basket.to_csv(processed_folder / 'market_basket.csv', index=False)

## 7. Classify product profitability

In [ ]:
product_profitability = (
    sales.groupby(['ProductKey', 'Product Name'], as_index=False)
    .agg(
        Revenue=('Sales USD', 'sum'),
        Profit=('Profit USD', 'sum'),
    )
)

product_profitability['Profit Margin %'] = (
    product_profitability['Profit'] / product_profitability['Revenue'] * 100
)

median_revenue = product_profitability['Revenue'].median()
median_profit = product_profitability['Profit'].median()
median_margin = product_profitability['Profit Margin %'].median()

def assign_profitability_group(row):
    if row['Revenue'] >= median_revenue and row['Profit Margin %'] < median_margin:
        return 'High Sales Low Profit'
    if row['Revenue'] < median_revenue and row['Profit'] >= median_profit:
        return 'High Profit Low Sales'
    return 'Other Products'

product_profitability['Profitability Group'] = product_profitability.apply(
    assign_profitability_group,
    axis=1,
)

product_profitability[['Revenue', 'Profit', 'Profit Margin %']] = (
    product_profitability[['Revenue', 'Profit', 'Profit Margin %']].round(2)
)

product_profitability.to_csv(processed_folder / 'product_profitability.csv', index=False)

## 8. Display the Phase 6 outputs

In [ ]:
print('Forecast rows:', len(sales_forecast))
print('Market-basket pairs:', len(market_basket))
print('Products classified:', len(product_profitability))
print()
print('Top product pairs:')
display(market_basket.head(10))
print('Profitability groups:')
display(product_profitability['Profitability Group'].value_counts())